⚡ Intermédiaire | ⏱ 45 min | 🔑 diagramme de classes, relations, Mermaid, SRP, dataclass, Strategy, Factory

# Exercices - UML et conception

Cours associé : [09-uml-conception.ipynb](09-uml-conception.ipynb).

## Comment travailler

Chaque exercice reproduit **un geste montré dans le cours**, dans un cas très proche. Pour chacun :

1. Lis l'**objectif** et les **étapes**.
2. Dans la cellule de code, remplace chaque `...` par ton code. Les noms des classes, méthodes et variables sont déjà écrits : garde-les, la vérification en a besoin.
3. Exécute ta cellule (`Shift + Entrée`) et compare avec le **résultat attendu**.
4. Exécute la cellule **Vérification** juste en dessous. Si elle affiche `OK`, c'est bon. Sinon, le message te dit ce qui ne va pas.

Si tu bloques, la ligne 📖 te dit quelle partie du cours relire.


## 1. Diagramme de classes

### 1.1 Du diagramme au code

**Objectif** : Implémenter exactement ce que décrit un diagramme.

```mermaid
classDiagram
    class Produit {
        +str nom
        -float prix
        +float TVA$
        +prix_ttc() float
    }
```

**À faire** :
1. `TVA = 0.2` est un attribut de classe.
2. `nom` est public, `prix` est privé : stocké dans `_prix`.
3. `prix_ttc()` renvoie `_prix * (1 + TVA)`.

**Résultat attendu** :
```
120.0
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 1. UML : le diagramme de classes », cellule « Le code qui correspond au diagramme ».


In [ ]:
class Produit:
    ...            # TVA = 0.2 : attribut de classe

    def __init__(self, nom, prix):
        ...        # nom : public
        ...        # prix : privé, stocké dans self._prix

    def prix_ttc(self):
        ...        # self._prix * (1 + TVA)

p = Produit("Clavier", 100)
print(p.prix_ttc())


In [ ]:
# Vérification
assert Produit.TVA == 0.2 and "TVA" not in vars(p), "TVA doit être un attribut de classe"
assert vars(p) == {"nom": "Clavier", "_prix": 100}, f"état attendu nom public et _prix privé, obtenu {vars(p)}"
assert abs(p.prix_ttc() - 120.0) < 1e-9
print("OK")


### 1.2 Du code au diagramme

**Objectif** : Décrire une classe existante en Mermaid.

**À faire** :
1. Lis la classe `Capteur` ci-dessous.
2. Complète la chaîne `mermaid` : une ligne par attribut avec sa visibilité (`+` ou `-`) et son type, une ligne par méthode. L'attribut de classe se termine par `$`.
3. Garde exactement les noms du code.

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 1. UML : le diagramme de classes », bloc Mermaid de `CompteBancaire`.


In [ ]:
class Capteur:
    UNITE = "°C"

    def __init__(self, nom):
        self.nom = nom
        self._mesures = []

    def ajouter(self, valeur):
        self._mesures.append(valeur)

    def moyenne(self):
        return sum(self._mesures) / len(self._mesures)

mermaid = """
classDiagram
    class Capteur {
        ...   %% attribut public nom, de type str
        ...   %% attribut privé mesures, de type list
        ...   %% attribut de classe UNITE, de type str : se termine par $
        ...   %% méthode ajouter(valeur)
        ...   %% méthode moyenne(), renvoie float
    }
"""
print(mermaid)


In [ ]:
# Vérification
lignes = [l.split("%%")[0].strip() for l in mermaid.splitlines() if l.split("%%")[0].strip()]
assert lignes[0] == "classDiagram" and "class Capteur {" in lignes, "le bloc doit déclarer class Capteur"
assert any(l.startswith("+") and l.endswith(" nom") for l in lignes), "nom est public : +str nom"
assert any(l.startswith("-") and "mesures" in l for l in lignes), "_mesures est privé : -list mesures"
assert any("UNITE$" in l for l in lignes), "l'attribut de classe se note UNITE$"
assert any(l.startswith("+ajouter(") for l in lignes) and any(l.startswith("+moyenne(") for l in lignes), "deux méthodes publiques"
print("OK")


## 2. Relations

### 2.1 Nommer la relation

**Objectif** : Reconnaître la relation à partir de la phrase.

**À faire** :
1. Pour chaque phrase, remplace `...` par l'un des mots : `"héritage"`, `"composition"`, `"agrégation"`, `"association"`.

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 2. Relations entre classes », tableau des relations.


In [ ]:
# Réponses possibles : "héritage", "composition", "agrégation", "association"
relations = {
    "Un Carre est une Forme": ...,                                                   # indice : « est une »
    "Une Maison a des Pieces, construites avec elle et détruites avec elle": ...,    # indice : même cycle de vie
    "Une Playlist regroupe des Morceaux qui existent sans elle": ...,                # indice : les parties survivent au tout
    "Un Medecin suit des Patients": ...,                                             # indice : ni « est un », ni « a un »
    "Un Manager est un Employe": ...,                                                # indice : « est un »
    "Un Ordinateur contient une CarteMere, fabriquée pour lui": ...,                 # indice : même cycle de vie
}


In [ ]:
# Vérification
attendu = {
    "Un Carre est une Forme": "héritage",
    "Une Maison a des Pieces, construites avec elle et détruites avec elle": "composition",
    "Une Playlist regroupe des Morceaux qui existent sans elle": "agrégation",
    "Un Medecin suit des Patients": "association",
    "Un Manager est un Employe": "héritage",
    "Un Ordinateur contient une CarteMere, fabriquée pour lui": "composition",
}
for phrase, rel in attendu.items():
    assert relations[phrase] == rel, f"{phrase!r} : attendu {rel!r}, obtenu {relations[phrase]!r}"
print("OK")


### 2.2 Composition et agrégation dans le code

**Objectif** : Traduire les deux losanges en Python.

**À faire** :
1. `Voiture` **compose** un `Moteur` : elle le crée elle-même dans `__init__`.
2. `Equipe` **agrège** des `Joueur` : elle reçoit une liste de joueurs déjà créés et la garde telle quelle (les mêmes objets).

**Résultat attendu** :
```
True True
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 2. Relations entre classes », cellule « Du diagramme au code ».


In [ ]:
class Moteur:
    pass

class Joueur:
    def __init__(self, nom):
        self.nom = nom

class Voiture:
    def __init__(self):
        ...        # composition : crée self.moteur = Moteur()

class Equipe:
    def __init__(self, joueurs):
        ...        # agrégation : garde les joueurs reçus dans self.joueurs

ana = Joueur("Ana")
equipe = Equipe([ana])
print(isinstance(Voiture().moteur, Moteur), equipe.joueurs[0] is ana)


In [ ]:
# Vérification
v1, v2 = Voiture(), Voiture()
assert isinstance(v1.moteur, Moteur) and v1.moteur is not v2.moteur, "chaque voiture crée son propre moteur"
assert equipe.joueurs[0] is ana, "l'équipe doit garder les joueurs reçus, pas des copies"
print("OK")


## 3. Du cahier des charges au diagramme

### 3.1 Noms, données, verbes

**Objectif** : Extraire classes, attributs et méthodes d'un texte.

**Texte** : « Un **parking** accueille des **voitures**. Chaque voiture a une plaque et une heure d'arrivée. Le parking a une capacité ; il peut faire entrer une voiture (si la capacité le permet) et la faire sortir, ce qui calcule le prix à payer. »

**À faire** :
1. Remplis `modele` : une clé par classe, avec la liste de ses `attributs` et de ses `methodes` (noms en `snake_case`).

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 3. Du cahier des charges au diagramme ».


In [ ]:
modele = {
    "Voiture": {"attributs": [...], "methodes": []},        # les deux données d'une voiture
    "Parking": {"attributs": [...], "methodes": [...]},     # capacité + ce qu'il contient ; les deux verbes
}


In [ ]:
# Vérification
assert set(modele) == {"Voiture", "Parking"}, "deux classes : Voiture et Parking"
assert "plaque" in modele["Voiture"]["attributs"], "une voiture a une plaque"
assert any("arriv" in a for a in modele["Voiture"]["attributs"]), "une voiture a une heure d'arrivée"
assert "capacite" in modele["Parking"]["attributs"], "le parking a une capacité"
assert any("voiture" in a for a in modele["Parking"]["attributs"]), "le parking accueille des voitures : un attribut qui les contient"
methodes = set(modele["Parking"]["methodes"])
assert any("entr" in m for m in methodes) and any("sort" in m for m in methodes), "deux verbes : entrer, sortir"
print("OK")


## 4. Une classe, une responsabilité

### 4.1 Découper une classe qui fait tout

**Objectif** : Séparer les trois responsabilités d'un rapport.

**À faire** :
1. `Rapport` ne garde que les données et `total()`.
2. `ExportCSV.exporter(rapport)` renvoie la ligne `"titre;total"`.
3. `ServiceMail.envoyer(rapport, destinataire)` renvoie `"mail à <destinataire> : <titre>"`.

**Résultat attendu** :
```
Ventes;60
mail à dir@ex.fr : Ventes
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 4. Une classe, une responsabilité », cellule « ✅ Séparer ».


In [ ]:
class Rapport:
    def __init__(self, titre, montants):
        self.titre = titre
        self.montants = montants

    def total(self):
        return sum(self.montants)

    def exporter_csv(self):                          # ❌ à déplacer dans ExportCSV.exporter(rapport)
        return f"{self.titre};{self.total()}"

    def envoyer_mail(self, destinataire):            # ❌ à déplacer dans ServiceMail.envoyer(rapport, destinataire)
        return f"mail à {destinataire} : {self.titre}"

class ExportCSV:
    ...        # méthode exporter(rapport) -> "titre;total"

class ServiceMail:
    ...        # méthode envoyer(rapport, destinataire) -> "mail à <destinataire> : <titre>"

r = Rapport("Ventes", [10, 20, 30])
print(ExportCSV().exporter(r))
print(ServiceMail().envoyer(r, "dir@ex.fr"))


In [ ]:
# Vérification
assert ExportCSV().exporter(r) == "Ventes;60"
assert ServiceMail().envoyer(r, "dir@ex.fr") == "mail à dir@ex.fr : Ventes"
assert not hasattr(r, "exporter_csv") and not hasattr(r, "envoyer_mail"), "Rapport ne doit plus exporter ni envoyer"
assert r.total() == 60
print("OK")


## 5. `@dataclass`

### 5.1 Générer `__init__`, `__repr__`, `__eq__`

**Objectif** : Écrire une classe de données sans boilerplate.

**À faire** :
1. Déclare `Eleve` comme `@dataclass` avec `nom: str`, `age: int` et `notes: list[float]` dont la valeur par défaut est une liste vide (`field(default_factory=list)`).
2. Ajoute la méthode `moyenne()` qui renvoie `0.0` si pas de notes, sinon la moyenne.

**Résultat attendu** :
```
Eleve(nom='Ana', age=20, notes=[12, 18]) True 15.0
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 5. Moins de code : `@dataclass` », cellule « @dataclass : __init__, __repr__ et __eq__ générés ».


In [ ]:
from dataclasses import dataclass, field

...            # décorateur dataclass
class Eleve:
    ...        # nom : str
    ...        # age : int
    ...        # notes : list[float], valeur par défaut field(default_factory=list)

    def moyenne(self):
        ...    # 0.0 si pas de notes, sinon la moyenne

a = Eleve("Ana", 20, [12, 18])
print(a, a == Eleve("Ana", 20, [12, 18]), a.moyenne())


In [ ]:
# Vérification
import dataclasses
assert dataclasses.is_dataclass(Eleve), "Eleve doit être une @dataclass"
assert repr(a) == "Eleve(nom='Ana', age=20, notes=[12, 18])", f"repr inattendu : {a!r}"
assert a == Eleve("Ana", 20, [12, 18]) and a.moyenne() == 15.0
b, c = Eleve("Bo", 21), Eleve("Cy", 22)
assert b.notes is not c.notes and b.moyenne() == 0.0, "chaque élève doit avoir sa propre liste vide"
print("OK")


### 5.2 Immuable avec `frozen=True`

**Objectif** : Obtenir un objet hachable et non modifiable.

**À faire** :
1. Déclare `Coord` comme `@dataclass(frozen=True)` avec `lat: float` et `lon: float`.

**Résultat attendu** :
```
1 FrozenInstanceError
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 5. Moins de code : `@dataclass` », cellule « frozen=True ».


In [ ]:
from dataclasses import dataclass

...            # décorateur dataclass avec frozen=True
class Coord:
    ...        # lat : float
    ...        # lon : float

villes = {Coord(43.6, 1.44), Coord(43.6, 1.44)}
try:
    Coord(0, 0).lat = 1
    erreur = "aucune"
except Exception as e:
    erreur = type(e).__name__
print(len(villes), erreur)


In [ ]:
# Vérification
assert len(villes) == 1, "deux Coord égales ne doivent compter qu'une fois dans le set"
assert erreur == "FrozenInstanceError", f"modifier un frozen doit lever FrozenInstanceError, obtenu {erreur}"
assert Coord(1, 2) == Coord(1, 2) and hash(Coord(1, 2)) == hash(Coord(1, 2))
print("OK")


## 6. Patrons de conception

### 6.1 Strategy

**Objectif** : Rendre le calcul de remise interchangeable.

**À faire** :
1. `Remise` est abstraite avec `appliquer(prix)` abstraite.
2. `SansRemise.appliquer` renvoie `prix` ; `Solde.appliquer` renvoie `prix * 0.7`.
3. `Article.prix_final()` délègue à `self.remise.appliquer(self.prix)`.

**Résultat attendu** :
```
100 70.0
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 6. Patrons de conception », cellule « Strategy ».


In [ ]:
from abc import ABC, abstractmethod

class Remise(ABC):
    ...            # décorateur qui rend appliquer() abstraite
    def appliquer(self, prix):
        ...        # pas de corps : laisse ... tel quel

class SansRemise(Remise):
    def appliquer(self, prix):
        ...        # renvoie prix

class Solde(Remise):
    def appliquer(self, prix):
        ...        # renvoie prix * 0.7

class Article:
    def __init__(self, prix, remise):
        self.prix = prix
        self.remise = remise

    def prix_final(self):
        ...        # délègue à self.remise.appliquer(self.prix)

a = Article(100, SansRemise())
avant = a.prix_final()
a.remise = Solde()
print(avant, a.prix_final())


In [ ]:
# Vérification
assert (avant, a.prix_final()) == (100, 70.0), f"obtenu {(avant, a.prix_final())}"
try:
    Remise()
except TypeError:
    pass
else:
    raise AssertionError("Remise doit être abstraite")
assert issubclass(Solde, Remise) and issubclass(SansRemise, Remise)
print("OK")


### 6.2 Factory

**Objectif** : Centraliser la création d'objets selon une chaîne.

**À faire** :
1. `creer_forme(nom, taille)` renvoie `Carre(taille)` pour `"carre"`, `Disque(taille)` pour `"disque"`.
2. Pour tout autre nom, elle lève `ValueError(f"forme inconnue : {nom}")`.
3. Utilise un dict `FORMES` qui associe le nom à la classe, comme dans le cours.

**Résultat attendu** :
```
Carre Disque forme inconnue : triangle
```

📖 Cours [09-uml-conception.ipynb](09-uml-conception.ipynb), section « 6. Patrons de conception », cellule « Factory ».


In [ ]:
class Carre:
    def __init__(self, cote):
        self.cote = cote

class Disque:
    def __init__(self, rayon):
        self.rayon = rayon

FORMES = {...}        # nom -> classe : "carre" et "disque"

def creer_forme(nom, taille):
    ...        # FORMES[nom](taille) ; si KeyError : raise ValueError(f"forme inconnue : {nom}") from None

try:
    erreur = creer_forme("triangle", 1)
except ValueError as e:
    erreur = e
print(type(creer_forme("carre", 2)).__name__, type(creer_forme("disque", 1)).__name__, erreur)


In [ ]:
# Vérification
c = creer_forme("carre", 2)
assert isinstance(c, Carre) and c.cote == 2
assert isinstance(creer_forme("disque", 1), Disque)
assert FORMES == {"carre": Carre, "disque": Disque}
assert isinstance(erreur, ValueError) and str(erreur) == "forme inconnue : triangle"
print("OK")


## Bravo

Fin du module 02-POO. Pour réviser : [CHEATSHEET-poo.ipynb](CHEATSHEET-poo.ipynb). Pour valider : [exercices/exercice-03-poo-projet.ipynb](../exercices/exercice-03-poo-projet.ipynb) puis [briefs/brief-02-gestionnaire-bibliotheque.ipynb](../briefs/brief-02-gestionnaire-bibliotheque.ipynb). Ensuite, le dossier `04-Bibliotheque-Standard`.
